# Parse SD monthly voter registration report

Uses:
- `requests` to get the PDF from [the Secretary of State's website](https://sdsos.gov/elections-voting/upcoming-elections/voter-registration-totals/voter-registration-by-county.aspx)
- `pdfplumber` to parse and validate the data

In [1]:
from io import BytesIO
from datetime import datetime
import csv
from pathlib import Path
import math

import requests
import pdfplumber

In [2]:
# fetch the PDF
url = "https://sdsos.gov/elections-voting/upcoming-elections/voter-registration-totals/2026%20Voter%20Registration/StatewideVotersByCounty_10.2.2026.pdf"
req = requests.get(url)

In [3]:
# open the pdf
pdf = pdfplumber.open(BytesIO(req.content))

In [4]:
# assumes a 2-page PDF -- validate this
assert len(pdf.pages) == 2

In [5]:
table_data = []

for page in pdf.pages:
    table_data.extend(page.extract_table())

date_row = table_data.pop(0)
header_row = table_data.pop(0)

In [6]:
table_data

[['Aurora', '443', '3', '1,018', '231', '3', '100', '71', '1,798'],
 ['Beadle', '2,230', '28', '5,421', '1,325', '46', '1,214', '694', '10,264'],
 ['Bennett', '647', '7', '733', '200', '2', '205', '124', '1,794'],
 ['Bon Homme', '777', '10', '2,480', '468', '', '230', '248', '3,965'],
 ['Brookings',
  '4,858',
  '94',
  '10,697',
  '3,568',
  '22',
  '2,816',
  '1280',
  '22,055'],
 ['Brown', '5,772', '83', '13,194', '2,978', '7', '1,854', '2733', '23,888'],
 ['Brule', '795', '13', '1,833', '419', '2', '220', '181', '3,282'],
 ['Buffalo', '514', '4', '165', '86', '4', '116', '53', '889'],
 ['Butte', '709', '54', '4,774', '810', '22', '668', '981', '7,037'],
 ['Campbell', '89', '2', '906', '63', '', '50', '75', '1,110'],
 ['Charles Mix', '1,665', '7', '2,851', '573', '', '382', '292', '5,478'],
 ['Clark', '449', '4', '1,822', '240', '', '181', '148', '2,696'],
 ['Clay', '2,689', '37', '3,144', '1,396', '20', '1,096', '1058', '8,382'],
 ['Codington',
  '3,509',
  '59',
  '10,857',
  '2,5

In [7]:
# extract the report date and column headers from the first line on the first page
report_date = datetime.strptime(
    date_row[0].split("-")[-1].strip(),
    '%B %d, %Y'
).date().isoformat()

print(report_date)

2026-10-02


In [8]:
# standardize column names
column_fixes = {
    'oth': 'other',
    'no party\naffiliation': 'npa'
}

headers = [column_fixes.get(x.lower(), x.lower()) for x in header_row if not x.lower().startswith("total")]
print(headers)

['county', 'democratic', 'libertarian', 'republican', 'independent', 'other', 'npa', 'inactive']


In [9]:
# lop off the cruft from the bottom of the table and format as a list of dicts
parsed_data = []

for row in table_data:
    if row[0] == "Total":
        break

    # remove the "total active" line
    row.pop()

    county_data = {"date": report_date} | {
        key: int(val.replace(",", "")) if val and key != "county" else val 
        for key, val in dict(zip(headers, row)).items()
    }

    parsed_data.append(county_data)    

In [10]:
parsed_data[0]

{'date': '2026-10-02',
 'county': 'Aurora',
 'democratic': 443,
 'libertarian': 3,
 'republican': 1018,
 'independent': 231,
 'other': 3,
 'npa': 100,
 'inactive': 71}

In [11]:
# check: 66 counties in SD
assert len(parsed_data) == 66

In [12]:
# save the PDF
outpath = Path('pdfs') / f'{report_date}.pdf'

with open(outpath, 'wb') as outfile:
    outfile.write(req.content)

print(f'Wrote {outpath}')

Wrote pdfs/2026-10-02.pdf


In [13]:
# write to file
filepath = (Path('data') / f'{report_date}.csv').resolve()

with open(filepath, 'w') as outfile:
    writer = csv.DictWriter(
        outfile,
        fieldnames=["date"] + headers
    )

    writer.writeheader()
    writer.writerows(parsed_data)

    print(f'Wrote file: {filepath}')

Wrote file: /Users/cjwinchester/south-dakota-voter-registration-data/data/2026-10-02.csv


In [15]:
%run build_files.ipynb

Wrote README.md (188 snapshots, updated October 5, 2026)
